In [1]:
# import libraries
from pydantic import BaseModel
from openai import AsyncOpenAI
import asyncio
from dotenv import load_dotenv
import os
import json
from sklearn.metrics import classification_report

In [2]:
# initialize empty dataset list
training_data = []
validation_data = []

with open("../../../01_data/training_validation_set/corrected_version/training_set.json", "r") as f:
    raw_training_data = json.load(f)

with open("../../../01_data/training_validation_set/corrected_version/validation_set.json", "r") as f:
    raw_val_data = json.load(f)

tag_to_label = {
    "pos": "positive",
    "neutral": "neutral",
    "neg": "negative"
}

# extract all annotations individually and append to training and validation data
for task in raw_training_data:
    if task["annotations"]:
        sentence = task["sentence"]
        for ann in task["annotations"]:
            social_group = ann["text"]
            stance = tag_to_label[ann["tag"][3:]]
            training_data.append({"sentence": sentence,
                                  "social_group": social_group,
                                  "stance": stance})


for task in raw_val_data:
    if task["annotations"]:
        sentence = task["sentence"]
        for ann in task["annotations"]:
            social_group = ann["text"]
            stance = tag_to_label[ann["tag"][3:]]
            validation_data.append({"sentence": sentence,
                                  "social_group": social_group,
                                  "stance": stance})

In [3]:
# compile prompt templates
medium = """
## Task Objective
Label the following sentence according to the stance the speaker expresses towards the highlighted social group. Return this label as a JSON object.
The stance can be either positive, neutral or negative. Label the sentence according to the following definition and criteria.

## Positive Stance
The text is positive towards the group if it expresses some sort of support or positive affect.
Especially within questions, a positive stance can also be expressed indirectly by raising the interests of the respective group or criticizing their disadvantage.

## Neutral Stance
The text is neutral towards the group if it references the group, but neither a positive nor negative stance are taken.
This happens mostly if the speaker mentions the group by reciting a neutral fact or the group is part of a title or an organization's name.

## Negative Stance
The text is negative towards the group if it expresses any form of critical or negative feeling towards the mentioned group. Raising awareness of the disadvantage faced by a group is not a negative but a positive stance.
"""

short = """
## Task Objective
Label the following sentence according to the stance the speaker expresses towards the highlighted social group. Return this label as a JSON object.
The stance can be either positive, neutral or negative.

## Positive Stance
The sentence expesses support or positive affect towards the group. Especially within questions, a positive stance can also be expressed indirectly by raising the interests of the group.

## Neutral Stance
The sentence references the group, but neither a positive nor negative stance are taken. This happens mostly within factual statements.

## Negative Stance
The sentence expresses a critical or negative feeling towards the mentioned group. Raising awareness of the disadvantage faced by a group is not a negative but a positive stance.
"""


# compile manual few-shot examples in json format
positive_example = {"text": "Stance towards young people in: Our party stands for improving the job opportunities of young people.",
        "llm_text": '{"stance": "positive"}'}
        
subtle_positive_example_1 = {"text": "Stance towards pupils: What is the Government's approach on creating a good working environment for pupils and teachers in our schools?",
     "llm_text": '{"stance": "positive"}'}

subtle_positive_example_2 = {"text": "Stance towards women: Women experience discrimination repeatedly throughout their lives",
     "llm_text": '{"stance": "positive"}'}

neutral_example = {"text": "Stance towards GP: Each person in this country should have the chance to get an appointment at his or her GP within a couple of days.",
     "llm_text": '{"stance": "neutral"}'}

negative_example = {"text": "Stance towards criminal offenders: We must do everything we can to tackle violence on our streets and get criminal offenders into jail.",
     "llm_text": '{"stance": "negative"}'}

# collect all in a list and mix up the order
few_shot_examples_all = [positive_example, neutral_example, subtle_positive_example_1, negative_example, subtle_positive_example_2]
few_shot_examples_subset = [subtle_positive_example_1, subtle_positive_example_2, neutral_example, negative_example]

In [4]:
# create prompt template
def compile_prompt_stance(system_prompt, few_shot_examples, test_sentence, social_group):

        chat = [
                {
                        "role": "system",
                        "content": system_prompt
                }
        ]

        # add all few-shot exampels
        for i in range(len(few_shot_examples)):
                chat.append({"role": "user", "content": f"Sentence: {few_shot_examples[i]['text']}"})
                chat.append({"role": "assistant", "content": few_shot_examples[i]["llm_text"]})
        
        # add the test sentence
        chat.append({"role": "user", "content": f"Stance towards {social_group} in: {test_sentence}"})

        return chat

In [5]:
# create dictionary storing rate limits
model_limits = {
    "gpt-4o-mini":
    {"token_limit": 200000,
     "request_limit": 500
    },
    "gpt-4o":
    {"token_limit": 30000,
     "request_limit": 500
    },
    "gpt-5-nano":
    {"token_limit": 200000,
     "request_limit": 500
    }
    }


# function to send out the request
async def send_request(client, model_name, prompt, output_class, temp=None, reasoning_effort=None):

    if model_name == "gpt-5-nano" :                                   
        response = await client.responses.parse(model=model_name,
                                                input=prompt,
                                                text_format=output_class,
                                                reasoning = {"effort": reasoning_effort}
                                                )
    elif model_name == "gpt-4o-mini":
        response = await client.responses.parse(model=model_name,
                                                input=prompt,
                                                text_format=output_class,
                                                temperature=temp
                                                )

    stance = response.output_parsed.stance
    try:
        if not isinstance(stance, str):
            return None
        return stance
    except Exception:
        return None
                                            
                                        
async def dispatch_all(client, model_name, system_message, few_shot_examples, output_class, validation_data, temp, reasoning_effort, safe_interval):
    tasks = []
    for row in validation_data:
        sentence = row["sentence"]
        social_group = row["social_group"]
        prompt = compile_prompt_stance(
            system_message,
            few_shot_examples,
            sentence,
            social_group
        )
        # create a task and fire it, do not wait
        task = asyncio.create_task(send_request(client, model_name, prompt, output_class, temp, reasoning_effort))
        tasks.append(task)
        
        # wait before starting the next request
        await asyncio.sleep(safe_interval)

    # gather all results once everything is started
    return await asyncio.gather(*tasks)

In [17]:
# select the model, system message and number of few shot examples
model_name = "gpt-5-nano"
system_message = medium

# set temperature and reasoning effort
temp = 0
reasoning_effort = "medium"

# empirically test a safe rate per minute
if model_name == "gpt-4o":
    safe_rpm = 50
else:
    safe_rpm = 100

# calculate a safe interval in which requests are sent
safe_interval = 60.0 / safe_rpm

# define class for the output
class StanceJSON(BaseModel):
    stance: str

# create client for interacting with API
load_dotenv()
client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# get random indices
#random.seed(0)
#random_indices = random.sample(range(len(validation_data)), 100)
#val_subset = [validation_data[i] for i in random_indices]
val_subset = validation_data
llm_output = await dispatch_all(client, model_name, system_message, few_shot_examples_all, StanceJSON, val_subset, temp, reasoning_effort, safe_interval)

In [18]:
for idx in range(0, len(val_subset)):
    print(val_subset[idx]["sentence"])
    print(val_subset[idx]["social_group"])
    print(f"Ground truth: {val_subset[idx]["stance"]}")
    print(f"LLM prediction: {llm_output[idx]}")
    print("-"*100)

Will not this arrangement protect the incomes of lower paid barristers?
lower paid barristers
Ground truth: positive
LLM prediction: positive
----------------------------------------------------------------------------------------------------
We have dealt with-and continue to deal with-abuse in the student visa system, which was allowed to increase significantly under the previous Labour Government, and non-EU migration is now at the levels of the late 1990s.
student
Ground truth: neutral
LLM prediction: neutral
----------------------------------------------------------------------------------------------------
The victims groups this week called on her to stand down and resign.
victims groups
Ground truth: neutral
LLM prediction: neutral
----------------------------------------------------------------------------------------------------
The Government are committed to improving station access for disabled people, including those with hidden disabilities.
disabled people
Ground truth:

In [8]:
results = {}

In [19]:
ground_truth = [item["stance"] for item in val_subset]
prediction = [stance for stance in llm_output]
metrics = classification_report(ground_truth, prediction, output_dict=True)
results["gpt-5-nano"] = {
           "negative_f1": metrics["negative"]["f1-score"],
           "neutral_f1": metrics["neutral"]["f1-score"],
           "positive_f1": metrics["positive"]["f1-score"],
           "macro_f1": metrics["macro avg"]["f1-score"]
           }

In [20]:
results

{'gpt-4o-mini': {'negative_f1': 0.33136094674556216,
  'neutral_f1': 0.5194174757281553,
  'positive_f1': 0.7485515643105446,
  'macro_f1': 0.533109995594754},
 'gpt-5-nano': {'negative_f1': 0.5747126436781609,
  'neutral_f1': 0.593320235756385,
  'positive_f1': 0.7806603773584906,
  'macro_f1': 0.6495644189310121}}

In [16]:
# export the preliminary results
with open("../eval_results/evaluation_metrics_gpt.json", "w") as f:
    json.dump(results, f)

In [ ]:
# set up temperatures and top
temperatures = [0, 0.25, 0.5]
reasoning_efforts = [None, "low", "medium", "high"]

# create client for interacting with API
load_dotenv()
client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# define class for the output
class StanceJSON(BaseModel):
    social_group: list[str]

# tune temperature value for 4o-mini model
model_name = "gpt-4o-mini"
system_message = short
few_shot_examples = 4
safe_rpm = 100
safe_interval = 60.0 / safe_rpm
results_4o_mini = {}
for temp in temperatures:
    llm_output = await dispatch_all(client, model_name, system_message, few_shot_examples_all, StanceJSON, validation_data, temp, None, safe_interval)
    ground_truth = [item["stance"] for item in validation_data]
    prediction = [stance for stance in llm_output]
    report = classification_report(ground_truth, prediction, output_dict=True)
    print(f"Report for temperature {temp}: {report}")
    results_4o_mini[temp: report]

# test all reasoning effort values for 5-nano
model_name = "gpt-5-nano"
system_message = medium
few_shot_examples = 4
safe_rpm = 100
safe_interval = 60.0 / safe_rpm
results_5_nano = {}
for re in reasoning_efforts:
    llm_output = await dispatch_all(client, model_name, system_message, few_shot_examples_all, StanceJSON, validation_data, None, reasoning_effort, safe_interval)
    ground_truth = [item["stance"] for item in validation_data]
    prediction = [stance for stance in llm_output]
    report = classification_report(ground_truth, prediction, output_dict=True)
    print(f"Report for temperature {temp}: {report}")
    results_5_nano[temp: report]